# 📘 기하학적 변환과 투시 변환

이미지의 크기, 방향, 관점을 변경하는 변환 기법을 배웁니다.

**학습 목표:**
- 확대/축소, 회전, 대칭
- 아핀 변환(이동, 회전, 스케일)
- 투시 변환(원근법 보정)
- 리맵핑

## 1. 기본 변환 — 확대/축소, 회전, 대칭

가장 기본적인 기하학적 변환입니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  확대/축소, 회전, 대칭                  │
# └─────────────────────────────────────────┘

# 테스트 이미지 생성 (화살표가 있는 이미지)
img = np.zeros((300, 300, 3), dtype=np.uint8)
img[:] = (200, 200, 200)  # 밝은 회색 배경
cv2.rectangle(img, (50, 50), (250, 250), (0, 100, 255), 3)
cv2.arrowedLine(img, (150, 30), (150, 270), (0, 0, 255), 3, tipLength=0.05)
cv2.putText(img, 'UP', (130, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 1. 확대/축소
img_scaled = cv2.resize(img_rgb, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_LINEAR)
img_shrink = cv2.resize(img_rgb, (150, 150), interpolation=cv2.INTER_AREA)

# 2. 회전
h, w = img_rgb.shape[:2]
M_rot = cv2.getRotationMatrix2D((w//2, h//2), 45, 1.0)  # 45도 회전
img_rotated = cv2.warpAffine(img_rgb, M_rot, (w, h))

# 3. 대칭
img_flip_h = cv2.flip(img_rgb, 1)  # 좌우 대칭
img_flip_v = cv2.flip(img_rgb, 0)   # 상하 대칭
img_flip_both = cv2.flip(img_rgb, -1)  # 상하좌우 대칭

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
titles = ['원본', '1.5배 확대', '0.5배 축소', '45도 회전', '좌우 대칭', '상하 대칭']
images = [img_rgb, img_scaled, img_shrink, img_rotated, img_flip_h, img_flip_v]
for ax, im, t in zip(axes.flat, images, titles):
    ax.imshow(im)
    ax.set_title(t)
    ax.axis('off')
plt.tight_layout()
plt.show()

print("💡 cv2.resize(): 보간법 지정 (INTER_LINEAR, INTER_AREA, INTER_CUBIC)")
print("💡 cv2.getRotationMatrix2D(): 중심점, 각도, 스케일 지정")
print("💡 cv2.flip(): 0=상하, 1=좌우, -1=상하좌우")

## 2. 아핀 변환과 투시 변환

**아핀 변환**(Affine)은 평행선을 유지하면서 변형합니다 (이동+회전+스케일+전단).
**투시 변환**(Perspective)은 원근 효과를 보정합니다 (문서 스캔 보정 등).

> 💡 아핀 변환: 3쌍의 대응점이 필요
> 💡 투시 변환: 4쌍의 대응점이 필요

In [ ]:
# ┌─────────────────────────────────────────┐
# │  아핀 변환과 투시 변환                   │
# └─────────────────────────────────────────┘

# 체크보드 패턴 이미지
img = np.zeros((300, 300, 3), dtype=np.uint8)
for i in range(0, 300, 30):
    for j in range(0, 300, 30):
        if (i // 30 + j // 30) % 2 == 0:
            img[i:i+30, j:j+30] = (180, 180, 180)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 아핀 변환 (3쌍 대응점)
rows, cols = img_rgb.shape[:2]
pts1 = np.float32([[50, 50], [200, 50], [50, 200]])
pts2 = np.float32([[10, 100], [200, 50], [100, 250]])

M_affine = cv2.getAffineTransform(pts1, pts2)
img_affine = cv2.warpAffine(img_rgb, M_affine, (cols, rows))

# 투시 변환 (4쌍 대응점)
pts1_p = np.float32([[56, 65], [236, 50], [28, 237], [210, 230]])
pts2_p = np.float32([[0, 0], [300, 0], [0, 300], [300, 300]])

M_perspective = cv2.getPerspectiveTransform(pts1_p, pts2_p)
img_perspective = cv2.warpPerspective(img_rgb, M_perspective, (300, 300))

# 결과 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_rgb); axes[0].set_title('원본 체크보드')
axes[1].imshow(img_affine); axes[1].set_title('아핀 변환')
axes[2].imshow(img_perspective); axes[2].set_title('투시 변환')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print("💡 아핀 변환: 평행선은 평행하게 유지 (이동+회전+스케일+전단)")
print("💡 투시 변환: 원근법 보정에 사용 (문서 스캔, 번호판 보정 등)")
print(f"\n아핀 변환 행렬:\n{M_affine}")
print(f"\n투시 변환 행렬:\n{M_perspective}")

## 🎯 연습 문제

1. 이미지를 90도, 180도, 270도 회전하는 세 가지 방법을 비교하세요.
2. 아핀 변환을 이용해 이미지를 x축 방향으로 30도 기울이세요 (전단 변환).
3. 투시 변환으로 비스듬히 찍힌 문서 이미지를 정면에서 본 것처럼 보정하세요.
4. `cv2.warpAffine()`과 `cv2.warpPerspective()`의 차이를 설명하세요.
5. 두 이미지를 좌우로 연결(stitch)하는 코드를 작성하세요.